In [1]:
# CATHEDRAL DIMENSIONLESS CORE v2.7
# Math and outputs only.
# From chaos to closure.
# Colab-safe: no class, no exponent operator.

import numpy as np
import math
import itertools
import json
import hashlib

pi = math.pi
s5 = math.sqrt(5.0)
phi = (1.0 + s5) / 2.0
sq = lambda x: x * x

D = 3
V = 12
N = 13
E = 30
F = 20
q = 5
G = 60

gamma = 1.0 / 81.0
dstar = (1.0 - gamma) * pi / (N * phi)
dcl = float(D) / F
Delta = dcl - dstar
Cmid = 0.5 * (dstar + dcl)

print("=" * 78)
print("CATHEDRAL DIMENSIONLESS CORE v2.7")
print("MATH AND OUTPUTS ONLY")
print("Chaos -> pi-phi-e flow -> G13 -> A5 -> hidden 4+4 -> outputs")
print("=" * 78)

print()
print("0. CORE PRIMITIVES")
print("  D, V, N, E, F, q, G       =", D, V, N, E, F, q, G)
print("  phi                       = %.15f" % phi)
print("  gamma = 1/81              = %.15f" % gamma)
print("  dstar = (1-gamma)pi/Nphi  = %.15f" % dstar)
print("  dcl = D/F                 = %.15f" % dcl)
print("  Delta = dcl-dstar         = %.15f" % Delta)
print("  Cmid                      = %.15f" % Cmid)


# ---------------------------------------------------------------------
# 1. ICOSAHEDRAL SHELL AND G13
# ---------------------------------------------------------------------

base = []
pairs = [(1.0, phi), (1.0, -phi), (-1.0, phi), (-1.0, -phi)]
for a, b in pairs:
    base.append((0.0, a, b))
    base.append((a, b, 0.0))
    base.append((b, 0.0, a))

verts = np.array(base)
diff = verts[:, None, :] - verts[None, :, :]
dist = np.sqrt((diff * diff).sum(-1))
mind = dist[dist > 1.0e-8].min()
A_shell = ((dist > 1.0e-8) & (np.abs(dist - mind) < 1.0e-8)).astype(float)

edges = []
i = 0
while i < 12:
    j = i + 1
    while j < 12:
        if A_shell[i, j] > 0.5:
            edges.append((i, j))
        j = j + 1
    i = i + 1

L_vertex = np.diag(A_shell.sum(1)) - A_shell
ev_v, Uv = np.linalg.eigh(L_vertex)

A13 = np.zeros((13, 13))
A13[1:13, 1:13] = A_shell
A13[0, 1:13] = 1.0
A13[1:13, 0] = 1.0
L13 = np.diag(A13.sum(1)) - A13
ev13 = np.linalg.eigvalsh(L13)

lambda_scalar = 0.0
lambda_triplet_low = 5.0 - s5
lambda_five_visible = 6.0
lambda_triplet_high = 5.0 + s5

print()
print("1. G13 GEOMETRY")
print("  shell vertices              =", len(verts))
print("  shell edges                 =", len(edges))
print("  shell degree                =", int(A_shell.sum(1)[0]))
print("  centered nodes              =", A13.shape[0])
print("  center degree               =", int(A13[0].sum()))
print("  shell Laplacian spectrum:")
vals_v, cnt_v = np.unique(np.round(ev_v, 6), return_counts=True)
for val, cnt in zip(vals_v, cnt_v):
    print("    %.6f multiplicity %d" % (val, cnt))
print("  centered G13 Laplacian spectrum:")
vals13, cnt13 = np.unique(np.round(ev13, 6), return_counts=True)
for val, cnt in zip(vals13, cnt13):
    print("    %.6f multiplicity %d" % (val, cnt))


# ---------------------------------------------------------------------
# 2. CHAOS RAIL AND PI-PHI-E FLOW
# ---------------------------------------------------------------------

def logistic(x, r):
    return r * x * (1.0 - x)

def get_cycle(r, transient, samples, tol):
    x = 0.5
    k = 0
    while k < transient:
        x = logistic(x, r)
        k = k + 1

    pts = []
    k = 0
    while k < samples:
        x = logistic(x, r)
        seen = False
        for u in pts:
            if abs(x - u) < tol:
                seen = True
        if not seen:
            pts.append(x)
        k = k + 1

    return sorted(pts)

def lyapunov_logistic(r, transient, samples):
    x = 0.3
    k = 0
    while k < transient:
        x = logistic(x, r)
        k = k + 1

    s = 0.0
    k = 0
    while k < samples:
        x = logistic(x, r)
        der = abs(r * (1.0 - 2.0 * x))
        if der < 1.0e-300:
            der = 1.0e-300
        s = s + math.log(der)
        k = k + 1

    return s / float(samples)

lo = 3.84151
hi = 3.8417002878419497
k = 0
while k < 38:
    mid = 0.5 * (lo + hi)
    cyc_mid = get_cycle(mid, 60000, 140, 1.0e-12)
    if len(cyc_mid) < 6:
        lo = mid
    else:
        if cyc_mid[0] > dstar:
            lo = mid
        else:
            hi = mid
    k = k + 1

r_pin = 0.5 * (lo + hi)
chaos_cycle = get_cycle(r_pin, 120000, 260, 1.0e-12)
chaos_lyap = lyapunov_logistic(r_pin, 8000, 40000)

def pi_phi_e_flow(target, steps, dt, seed):
    rng = np.random.RandomState(seed)
    x = rng.rand(13)
    s0 = float(x.std())

    k = 0
    while k < steps:
        t = k * dt
        lap = L13.dot(x)
        pull = (phi - 1.0) * math.exp(-t / 10.0) * (x - target) * (1.0 + x * x)
        x = x + dt * (-(1.0 / (4.0 * pi)) * lap - pull)
        k = k + 1

    return float(x.mean()), float(x.std()), s0

flow_mean, flow_std, flow_start_std = pi_phi_e_flow(dstar, 7000, 0.01, 1)

print()
print("2. CHAOS ORIGIN AND PI-PHI-E FLOW")
print("  period-3 onset r            = %.15f" % (1.0 + math.sqrt(8.0)))
print("  r_pin for frozen dstar      = %.15f" % r_pin)
print("  chaos rail 0                = %.15f" % chaos_cycle[0])
print("  chaos rail 1                = %.15f" % chaos_cycle[1])
print("  rail 0 error to dstar       = %.3e" % abs(chaos_cycle[0] - dstar))
print("  rail 1 error to dcl         = %.3e" % abs(chaos_cycle[1] - dcl))
print("  rail split                  = %.15f" % (chaos_cycle[1] - chaos_cycle[0]))
print("  Delta target                = %.15f" % Delta)
print("  logistic Lyapunov           = %.15f" % chaos_lyap)
print("  flow start spread           = %.15f" % flow_start_std)
print("  flow final spread           = %.15e" % flow_std)
print("  flow final mean             = %.15f" % flow_mean)
print("  flow error to dstar         = %.3e" % abs(flow_mean - dstar))


# ---------------------------------------------------------------------
# 3. A5 CHARACTER DECOMPOSITION
# ---------------------------------------------------------------------

class_sizes = np.array([1.0, 15.0, 20.0, 12.0, 12.0])

chars = {
    "1": np.array([1.0, 1.0, 1.0, 1.0, 1.0]),
    "3": np.array([3.0, -1.0, 0.0, phi, 1.0 - phi]),
    "3p": np.array([3.0, -1.0, 0.0, 1.0 - phi, phi]),
    "4": np.array([4.0, 0.0, 1.0, -1.0, -1.0]),
    "5": np.array([5.0, 1.0, -1.0, 0.0, 0.0])
}

def char_mult(perm_char, name):
    return (class_sizes * perm_char * chars[name]).sum() / 60.0

vertex_char = np.array([12.0, 0.0, 0.0, 2.0, 2.0])
face_char = np.array([20.0, 0.0, 2.0, 0.0, 0.0])
edge_char = np.array([30.0, 2.0, 0.0, 0.0, 0.0])

vertex_irreps = ["1", "3", "3p", "5"]
face_irreps = ["1", "3", "3p", "4", "4", "5"]
edge_irreps = ["1", "3", "3p", "4", "4", "5", "5", "5"]
hidden_irreps = ["4", "4"]

print()
print("3. A5 CHARACTER DECOMPOSITION")
for label, pc in [("V12 vertex", vertex_char), ("F20 face", face_char), ("E30 edge", edge_char)]:
    print("  " + label)
    for name in ["1", "3", "3p", "4", "5"]:
        print("    %-4s multiplicity %.6f" % (name, char_mult(pc, name)))
print("  V12 irreps                 =", vertex_irreps)
print("  F20 irreps                 =", face_irreps)
print("  F20 - V12 hidden sector    =", hidden_irreps)

a5_products = {}
a5_products[("1", "1")] = ["1"]
a5_products[("1", "3")] = ["3"]
a5_products[("1", "3p")] = ["3p"]
a5_products[("1", "4")] = ["4"]
a5_products[("1", "5")] = ["5"]
a5_products[("3", "3")] = ["1", "3", "5"]
a5_products[("3p", "3p")] = ["1", "3p", "5"]
a5_products[("3", "3p")] = ["4", "5"]
a5_products[("3", "4")] = ["3p", "4", "5"]
a5_products[("3p", "4")] = ["3", "4", "5"]
a5_products[("3", "5")] = ["3", "3p", "4", "5"]
a5_products[("3p", "5")] = ["3", "3p", "4", "5"]
a5_products[("4", "4")] = ["1", "3", "3p", "4", "5"]
a5_products[("4", "5")] = ["3", "3p", "4", "5", "5"]
a5_products[("5", "5")] = ["1", "3", "3p", "4", "4", "5", "5"]

keys_now = list(a5_products.keys())
for key in keys_now:
    a, b = key
    a5_products[(b, a)] = a5_products[key]

sym2_4 = ["1", "4", "5"]
wedge2_4 = ["3", "3p"]

print("  4 x 4 product              =", a5_products[("4", "4")])
print("  Sym2(4)                    =", sym2_4)
print("  Wedge2(4)                  =", wedge2_4)


# ---------------------------------------------------------------------
# 4. FACES, INCIDENCE, HIDDEN SPLIT
# ---------------------------------------------------------------------

faces = []
for tri in itertools.combinations(range(12), 3):
    a, b, c = tri
    if A_shell[a, b] > 0.5 and A_shell[a, c] > 0.5 and A_shell[b, c] > 0.5:
        faces.append(tri)

B = np.zeros((20, 12))
r = 0
while r < len(faces):
    tri = faces[r]
    for vtx in tri:
        B[r, vtx] = 1.0
    r = r + 1

rankB = np.linalg.matrix_rank(B, tol=1.0e-10)
hole_dim = 20 - rankB
BtB = B.T.dot(B)
BtB_target = 5.0 * np.eye(12) + 2.0 * A_shell
BtB_err = np.max(np.abs(BtB - BtB_target))

A_face = np.zeros((20, 20))
i = 0
while i < 20:
    j = i + 1
    while j < 20:
        shared = 0
        for vi in faces[i]:
            for vj in faces[j]:
                if vi == vj:
                    shared = shared + 1
        if shared == 2:
            A_face[i, j] = 1.0
            A_face[j, i] = 1.0
        j = j + 1
    i = i + 1

L_face = np.diag(A_face.sum(1)) - A_face
Utmp, Stmp, Vhtmp = np.linalg.svd(B.T, full_matrices=True)
rankBT = np.linalg.matrix_rank(B.T, tol=1.0e-10)
Qhole = Vhtmp.T[:, rankBT:]

Lhole = Qhole.T.dot(L_face).dot(Qhole)
Wh, Uh = np.linalg.eigh(Lhole)
Qsplit = Qhole.dot(Uh)

Q3 = Qsplit[:, np.abs(Wh - 3.0) < 1.0e-8]
Q5 = Qsplit[:, np.abs(Wh - 5.0) < 1.0e-8]
vals_h, cnt_h = np.unique(np.round(Wh, 6), return_counts=True)

print()
print("4. FACE INCIDENCE AND HIDDEN SPLIT")
print("  faces                       =", len(faces))
print("  rank(B)                     =", rankB)
print("  dim ker(B.T)                =", hole_dim)
print("  B.TB = 5I + 2A max error    = %.3e" % BtB_err)
print("  face-dual degree            =", int(A_face.sum(1)[0]))
print("  face-dual edges             =", int(A_face.sum() / 2.0))
print("  max B.T Qhole               = %.3e" % np.max(np.abs(B.T.dot(Qhole))))
print("  hidden spectrum:")
for val, cnt in zip(vals_h, cnt_h):
    print("    %.6f multiplicity %d" % (val, cnt))
print("  Q3 dim                      =", Q3.shape[1])
print("  Q5 dim                      =", Q5.shape[1])


# ---------------------------------------------------------------------
# 5. PROJECTORS AND RETURN SIGNATURES
# ---------------------------------------------------------------------

def spectral_projector(evals, evecs, target, tol):
    cols = []
    i = 0
    while i < len(evals):
        if abs(evals[i] - target) < tol:
            cols.append(i)
        i = i + 1
    Q = evecs[:, cols]
    return Q.dot(Q.T), len(cols)

P_scalar, dim_scalar = spectral_projector(ev_v, Uv, lambda_scalar, 1.0e-8)
P_triplet_low, dim_triplet_low = spectral_projector(ev_v, Uv, lambda_triplet_low, 1.0e-8)
P_five, dim_five = spectral_projector(ev_v, Uv, lambda_five_visible, 1.0e-8)
P_triplet_high, dim_triplet_high = spectral_projector(ev_v, Uv, lambda_triplet_high, 1.0e-8)
P_total = P_scalar + P_triplet_low + P_five + P_triplet_high

def sym_return(Q):
    cols = []
    i = 0
    while i < Q.shape[1]:
        j = i
        while j < Q.shape[1]:
            h = Q[:, i] * Q[:, j]
            if j > i:
                h = math.sqrt(2.0) * h
            cols.append(B.T.dot(h))
            j = j + 1
        i = i + 1
    return np.column_stack(cols)

def mixed_return(QA, QB):
    cols = []
    i = 0
    while i < QA.shape[1]:
        j = 0
        while j < QB.shape[1]:
            h = QA[:, i] * QB[:, j]
            cols.append(B.T.dot(h))
            j = j + 1
        i = i + 1
    return np.column_stack(cols)

def projector_power(P, M):
    X = P.dot(M)
    val = np.linalg.norm(X, "fro")
    return val * val

def canonical_signature(M, label):
    total = np.linalg.norm(M, "fro")
    total = total * total
    p_scalar = projector_power(P_scalar, M)
    p_low = projector_power(P_triplet_low, M)
    p_five = projector_power(P_five, M)
    p_high = projector_power(P_triplet_high, M)
    p_triplet = p_low + p_high

    ratio = 0.0
    if p_five > 1.0e-14:
        ratio = p_scalar / p_five

    chi_signed = 0.0
    chi_abs = 0.0
    if p_triplet > 1.0e-14:
        chi_signed = (p_low - p_high) / p_triplet
        chi_abs = abs(chi_signed)

    return {
        "label": label,
        "total": total,
        "scalar_power": p_scalar,
        "triplet_low_power": p_low,
        "five_power": p_five,
        "triplet_high_power": p_high,
        "triplet_total": p_triplet,
        "scalar_five_ratio": ratio,
        "chiral_signed": chi_signed,
        "chiral_abs": chi_abs
    }

M33 = sym_return(Q3)
M55 = sym_return(Q5)
M35 = mixed_return(Q3, Q5)
Mall = sym_return(Qsplit)

sig33 = canonical_signature(M33, "Sym2(4_3)")
sig55 = canonical_signature(M55, "Sym2(4_5)")
sig35 = canonical_signature(M35, "4_3 x 4_5")

print()
print("5. PROJECTORS AND RETURN SIGNATURES")
print("  visible projector dims       =", dim_scalar, dim_triplet_low, dim_five, dim_triplet_high)
print("  return ranks M33 M55 M35 all =", np.linalg.matrix_rank(M33), np.linalg.matrix_rank(M55), np.linalg.matrix_rank(M35), np.linalg.matrix_rank(Mall))
for sig in [sig33, sig55, sig35]:
    print("  " + sig["label"])
    print("    scalar power               = %.15f" % sig["scalar_power"])
    print("    five power                 = %.15f" % sig["five_power"])
    print("    triplet low power          = %.15f" % sig["triplet_low_power"])
    print("    triplet high power         = %.15f" % sig["triplet_high_power"])
    print("    scalar/five ratio          = %.15f" % sig["scalar_five_ratio"])
    print("    chiral signed              = %.15f" % sig["chiral_signed"])
    print("    chiral abs                 = %.15f" % sig["chiral_abs"])


# ---------------------------------------------------------------------
# 6. ROUTER
# ---------------------------------------------------------------------

def channel_role_from_signature(sig):
    scalar = sig["scalar_power"]
    five = sig["five_power"]
    trip = sig["triplet_total"]
    chi = sig["chiral_abs"]

    if scalar > 1.0e-12 and five > 1.0e-12 and trip < 1.0e-12:
        return "even_scalar_five_return"

    if trip > 1.0e-12 and scalar < 1.0e-12 and chi > 1.0e-12:
        return "odd_chiral_triplet_return"

    return "unrouted"

channels = [
    {"name": "Sym2(4_3)", "signature": sig33, "curvature": 3.0},
    {"name": "Sym2(4_5)", "signature": sig55, "curvature": 5.0},
    {"name": "4_3 x 4_5", "signature": sig35, "curvature": 4.0}
]

for ch in channels:
    ch["role"] = channel_role_from_signature(ch["signature"])

scalar_slot_candidates = []
for ch in channels:
    if ch["role"] == "even_scalar_five_return":
        scalar_slot_candidates.append(ch)

best_five = -1.0
for ch in scalar_slot_candidates:
    if ch["signature"]["five_power"] > best_five:
        best_five = ch["signature"]["five_power"]

scalar_slot_matches = []
for ch in scalar_slot_candidates:
    if abs(ch["signature"]["five_power"] - best_five) < 1.0e-12:
        scalar_slot_matches.append(ch)

scalar_slot = scalar_slot_matches[0]
W_scalar = scalar_slot["signature"]["scalar_five_ratio"]

chiral_slot_candidates = []
for ch in channels:
    if ch["role"] == "odd_chiral_triplet_return":
        chiral_slot_candidates.append(ch)

chiral_slot = chiral_slot_candidates[0]
chi_signed = chiral_slot["signature"]["chiral_signed"]

phi2 = phi * phi
phi3 = phi2 * phi
phi6 = phi3 * phi3

print()
print("6. A5 SECTOR ROUTER")
for ch in channels:
    print("  %-12s role=%s" % (ch["name"], ch["role"]))
print("  scalar slot channel          =", scalar_slot["name"])
print("  chiral slot channel          =", chiral_slot["name"])
print("  W_scalar                     = %.15f" % W_scalar)
print("  chi_signed                   = %.15f" % chi_signed)


# ---------------------------------------------------------------------
# 7. ACTION KERNEL AND OUTPUT SLOTS
# ---------------------------------------------------------------------

coherent_dim = float(D + 1)
entropy_dim = float(N) - coherent_dim
hidden_dim = float(Qhole.shape[1])
active_faces = float(F - 1)
face_plus_vacuum = float(F + 1)
matter_orientation = float(2 * D)

def stationary(base, source, curvature):
    return base + source / curvature

def oscillator_amplitude(curvature):
    return 1.0 / math.sqrt(curvature)

def quadratic_norm(source, curvature):
    return (source * source) / (curvature * curvature)

def multiplicative_screen(source, screen):
    return source * screen

face_screen_curvature = F - 1.0 / hidden_dim
entropy_scalar_second_source = Delta * Delta * entropy_dim

slot_scalar_base = N * N - E - 2.0
slot_scalar_source = Delta * (N + W_scalar - (1.0 + gamma) / D)
slot_scalar_bare_inverse = stationary(slot_scalar_base, slot_scalar_source, 1.0)
slot_scalar_entropy_dressing = stationary(0.0, entropy_scalar_second_source, q * N)
slot_scalar_screened_inverse = slot_scalar_bare_inverse - slot_scalar_entropy_dressing

slot_face_oscillator = oscillator_amplitude(face_screen_curvature)
slot_chiral_golden_leak = stationary(0.0, Delta * (phi6 - 1.0), 1.0)
slot_direct_spatial_gap = stationary(0.0, D * Delta, 2.0)
slot_entropy_area = stationary(0.0, gamma * Delta * (1.0 + W_scalar * gamma), 1.0)
slot_quadratic_area = quadratic_norm(slot_entropy_area, pi)
slot_matter_orientation_area = stationary(0.0, matter_orientation * slot_quadratic_area, 1.0)
slot_face_vacuum_area = stationary(0.0, face_plus_vacuum * slot_quadratic_area, 1.0)

slot_solar_triplet = stationary(1.0 / D, -V * Delta, 1.0)
slot_atmospheric_triplet = stationary(0.5, F * Delta + gamma, 1.0)
slot_reactor_entropy_pair = multiplicative_screen(2.0 * gamma * (dstar / dcl) * (dstar / dcl), 1.0 - F * Delta)
slot_pmns_area = -N * Delta

slot_matter_closure = matter_orientation / active_faces
slot_vacuum_closure = float(N) / active_faces
slot_visible_fraction = 2.0 / float(N)
slot_entropy_matter_fraction = (float(N) - 2.0) / float(N)
slot_baryon_closure = slot_matter_closure * slot_visible_fraction
slot_cdm_closure = slot_matter_closure * slot_entropy_matter_fraction
slot_dark_closure = slot_cdm_closure + slot_vacuum_closure

slot_tilt = stationary(1.0, -D * gamma + Delta, 1.0)
slot_running = -Delta * Delta
slot_tensor = 16.0 * gamma * Delta
slot_tensor_nt = -slot_tensor / 8.0
slot_face_screen_depth = stationary(1.0 / active_faces, Delta, pi)
slot_fivefold_growth = math.cos(pi / q)

slot_RG_flow = stationary(0.0, F * Delta + gamma, 1.0)
slot_lambda_H = 1.0 / hidden_dim + gamma / D
slot_yt_RG = 1.0 - slot_RG_flow
slot_yt_pole = 1.0 - Delta

alpha_inv_bare = slot_scalar_bare_inverse
alpha_inv = slot_scalar_screened_inverse
alpha = 1.0 / alpha_inv

sin2w = float(D) / float(N)
cos2w = 1.0 - sin2w
alpha_s = float(F) / float(N * N)

e2 = 4.0 * pi * alpha
e_coup = math.sqrt(e2)
g_coup = e_coup / math.sqrt(sin2w)
gp_coup = e_coup / math.sqrt(cos2w)

alpha1 = (5.0 / 3.0) * alpha / cos2w
alpha2 = alpha / sin2w
alpha3 = alpha_s

Vus = slot_face_oscillator
Vcb = slot_chiral_golden_leak
Vub = slot_direct_spatial_gap
J_ckm = slot_entropy_area

s12 = Vus
s23 = Vcb
s13 = Vub
c12 = math.sqrt(1.0 - s12 * s12)
c23 = math.sqrt(1.0 - s23 * s23)
c13 = math.sqrt(1.0 - s13 * s13)

sin_delta_ckm = J_ckm / (s12 * s23 * s13 * c12 * c23 * c13 * c13)
if sin_delta_ckm > 1.0:
    sin_delta_ckm = 1.0
if sin_delta_ckm < -1.0:
    sin_delta_ckm = -1.0

delta_ckm = math.asin(sin_delta_ckm)
ep = math.cos(delta_ckm) + 1j * math.sin(delta_ckm)
em = math.cos(delta_ckm) - 1j * math.sin(delta_ckm)

Vckm = np.array([
    [c12 * c13, s12 * c13, s13 * em],
    [-s12 * c23 - c12 * s23 * s13 * ep, c12 * c23 - s12 * s23 * s13 * ep, s23 * c13],
    [s12 * s23 - c12 * c23 * s13 * ep, -c12 * s23 - s12 * c23 * s13 * ep, c23 * c13]
], dtype=complex)

ckm_unit = np.max(np.abs(Vckm.conj().T.dot(Vckm) - np.eye(3)))

theta_qcd = slot_quadratic_area
eta_baryon = slot_matter_orientation_area
A_s = slot_face_vacuum_area
ln10As = math.log(10000000000.0 * A_s)

pmns_s12_sq = slot_solar_triplet
pmns_s23_sq = slot_atmospheric_triplet
pmns_s13_sq = slot_reactor_entropy_pair
pmns_J = slot_pmns_area

pmns_s12 = math.sqrt(pmns_s12_sq)
pmns_s23 = math.sqrt(pmns_s23_sq)
pmns_s13 = math.sqrt(pmns_s13_sq)

pmns_c12 = math.sqrt(1.0 - pmns_s12_sq)
pmns_c23 = math.sqrt(1.0 - pmns_s23_sq)
pmns_c13 = math.sqrt(1.0 - pmns_s13_sq)

pmns_Jmax = pmns_s12 * pmns_c12 * pmns_s23 * pmns_c23 * pmns_s13 * pmns_c13 * pmns_c13
pmns_sind = pmns_J / pmns_Jmax

if pmns_sind > 1.0:
    pmns_sind = 1.0
if pmns_sind < -1.0:
    pmns_sind = -1.0

branch_A = pi + math.asin(abs(pmns_sind))
branch_B = 2.0 * pi - math.asin(abs(pmns_sind))

if chi_signed > 0.0:
    delta_pmns = branch_B
    pmns_branch_name = "quadrant IV"
else:
    delta_pmns = branch_A
    pmns_branch_name = "quadrant III"

epL = math.cos(delta_pmns) + 1j * math.sin(delta_pmns)
emL = math.cos(delta_pmns) - 1j * math.sin(delta_pmns)

Upmns = np.array([
    [pmns_c12 * pmns_c13, pmns_s12 * pmns_c13, pmns_s13 * emL],
    [-pmns_s12 * pmns_c23 - pmns_c12 * pmns_s23 * pmns_s13 * epL, pmns_c12 * pmns_c23 - pmns_s12 * pmns_s23 * pmns_s13 * epL, pmns_s23 * pmns_c13],
    [pmns_s12 * pmns_s23 - pmns_c12 * pmns_c23 * pmns_s13 * epL, -pmns_c12 * pmns_s23 - pmns_s12 * pmns_c23 * pmns_s13 * epL, pmns_c23 * pmns_c13]
], dtype=complex)

pmns_unit = np.max(np.abs(Upmns.conj().T.dot(Upmns) - np.eye(3)))

r_nu = dstar + dstar * dstar
nu_dm_ratio = r_nu * r_nu

Omega_b = slot_baryon_closure
Omega_c = slot_cdm_closure
Omega_m = slot_matter_closure
Omega_L = slot_vacuum_closure
Omega_dark = slot_dark_closure

n_s = slot_tilt
running_ns = slot_running
tensor_r = slot_tensor
tensor_nt = slot_tensor_nt
tau_reio = slot_face_screen_depth
sigma8 = slot_fivefold_growth

x_RG = slot_RG_flow
lambda_H = slot_lambda_H
yt_RG = slot_yt_RG
yt_pole = slot_yt_pole


# ---------------------------------------------------------------------
# 8. FULL OUTPUT LEDGER
# ---------------------------------------------------------------------

print()
print("7. FULL OBSERVABLE OUTPUTS")

print()
print("ELECTROMAGNETIC AND GAUGE")
print("  alpha inverse bare          = %.15f" % alpha_inv_bare)
print("  alpha inverse dressed       = %.15f" % alpha_inv)
print("  alpha                       = %.15e" % alpha)
print("  e coupling                  = %.15f" % e_coup)
print("  sin2 thetaW                 = %.15f" % sin2w)
print("  cos2 thetaW                 = %.15f" % cos2w)
print("  g weak coupling             = %.15f" % g_coup)
print("  gp hypercharge coupling     = %.15f" % gp_coup)
print("  alpha_s                     = %.15f" % alpha_s)
print("  alpha1                      = %.15e" % alpha1)
print("  alpha2                      = %.15e" % alpha2)
print("  alpha3                      = %.15e" % alpha3)
print("  alpha1 inverse              = %.15f" % (1.0 / alpha1))
print("  alpha2 inverse              = %.15f" % (1.0 / alpha2))
print("  alpha3 inverse              = %.15f" % (1.0 / alpha3))

print()
print("CKM AND QUARK MIXING")
print("  Vus                         = %.15f" % Vus)
print("  Vcb                         = %.15f" % Vcb)
print("  Vub                         = %.15f" % Vub)
print("  J_CKM                       = %.15e" % J_ckm)
print("  sin delta_CKM               = %.15f" % sin_delta_ckm)
print("  delta_CKM radians           = %.15f" % delta_ckm)
print("  delta_CKM degrees           = %.15f" % (delta_ckm * 180.0 / pi))
print("  CKM unitarity error         = %.3e" % ckm_unit)
print("  abs CKM matrix:")
print(np.abs(Vckm))

print()
print("CP, BARYOGENESIS, SCALAR AMPLITUDE")
print("  theta_QCD                   = %.15e" % theta_qcd)
print("  eta_B                       = %.15e" % eta_baryon)
print("  A_s                         = %.15e" % A_s)
print("  ln(1e10 A_s)                = %.15f" % ln10As)

print()
print("PMNS AND NEUTRINO MIXING")
print("  sin2 theta12                = %.15f" % pmns_s12_sq)
print("  sin2 theta23                = %.15f" % pmns_s23_sq)
print("  sin2 theta13                = %.15f" % pmns_s13_sq)
print("  theta12 degrees             = %.15f" % (math.asin(pmns_s12) * 180.0 / pi))
print("  theta23 degrees             = %.15f" % (math.asin(pmns_s23) * 180.0 / pi))
print("  theta13 degrees             = %.15f" % (math.asin(pmns_s13) * 180.0 / pi))
print("  J_PMNS                      = %.15f" % pmns_J)
print("  J_PMNS max                  = %.15f" % pmns_Jmax)
print("  sin delta_PMNS              = %.15f" % pmns_sind)
print("  delta_PMNS radians          = %.15f" % delta_pmns)
print("  delta_PMNS degrees          = %.15f" % (delta_pmns * 180.0 / pi))
print("  PMNS branch                 =", pmns_branch_name)
print("  PMNS unitarity error        = %.3e" % pmns_unit)
print("  abs PMNS matrix:")
print(np.abs(Upmns))

print()
print("NEUTRINO MASS LADDER")
print("  r_nu = dstar + dstar*dstar  = %.15f" % r_nu)
print("  dm21/dm31 seed ratio        = %.15f" % nu_dm_ratio)

print()
print("DARK SECTOR AND COSMOLOGICAL CLOSURE")
print("  Omega_b                     = %.15f" % Omega_b)
print("  Omega_c                     = %.15f" % Omega_c)
print("  Omega_m                     = %.15f" % Omega_m)
print("  Omega_Lambda                = %.15f" % Omega_L)
print("  Omega_dark                  = %.15f" % Omega_dark)
print("  Omega_c / Omega_b           = %.15f" % (Omega_c / Omega_b))
print("  Omega_dark / Omega_b        = %.15f" % (Omega_dark / Omega_b))
print("  closure sum b+c+Lambda      = %.15f" % (Omega_b + Omega_c + Omega_L))

print()
print("INFLATION AND GROWTH")
print("  n_s                         = %.15f" % n_s)
print("  running_ns                  = %.15e" % running_ns)
print("  tensor_r                    = %.15e" % tensor_r)
print("  tensor_nt                   = %.15e" % tensor_nt)
print("  tau_reio                    = %.15f" % tau_reio)
print("  sigma8                      = %.15f" % sigma8)

print()
print("RG, HIGGS, TOP")
print("  x_RG                        = %.15f" % x_RG)
print("  lambda_H                    = %.15f" % lambda_H)
print("  y_t_RG                      = %.15f" % yt_RG)
print("  y_t_pole                    = %.15f" % yt_pole)

print()
print("ABSTRACT ROUTER SLOTS")
print("  scalar bare inverse         = %.15f" % slot_scalar_bare_inverse)
print("  scalar screened inverse     = %.15f" % slot_scalar_screened_inverse)
print("  scalar entropy dressing     = %.15e" % slot_scalar_entropy_dressing)
print("  face oscillator             = %.15f" % slot_face_oscillator)
print("  chiral golden leak          = %.15f" % slot_chiral_golden_leak)
print("  direct spatial gap          = %.15f" % slot_direct_spatial_gap)
print("  entropy CP area             = %.15e" % slot_entropy_area)
print("  quadratic CP norm           = %.15e" % slot_quadratic_area)
print("  solar triplet slot          = %.15f" % slot_solar_triplet)
print("  atmospheric triplet slot    = %.15f" % slot_atmospheric_triplet)
print("  reactor entropy pair        = %.15f" % slot_reactor_entropy_pair)
print("  leptonic area slot          = %.15f" % slot_pmns_area)
print("  baryon closure              = %.15f" % slot_baryon_closure)
print("  CDM closure                 = %.15f" % slot_cdm_closure)
print("  matter closure              = %.15f" % slot_matter_closure)
print("  vacuum closure              = %.15f" % slot_vacuum_closure)
print("  dark closure                = %.15f" % slot_dark_closure)

freeze = {
    "name": "Cathedral Dimensionless Core v2.7 math outputs",
    "D": D,
    "V": V,
    "N": N,
    "E": E,
    "F": F,
    "q": q,
    "G": G,
    "phi": phi,
    "gamma": gamma,
    "dstar": dstar,
    "dcl": dcl,
    "Delta": Delta,
    "r_pin": r_pin,
    "chaos_cycle_0": chaos_cycle[0],
    "chaos_cycle_1": chaos_cycle[1],
    "chaos_lyapunov": chaos_lyap,
    "flow_mean": flow_mean,
    "flow_std": flow_std,
    "vertex_irreps": vertex_irreps,
    "face_irreps": face_irreps,
    "hidden_irreps": hidden_irreps,
    "scalar_slot_channel": scalar_slot["name"],
    "chiral_slot_channel": chiral_slot["name"],
    "W_scalar": W_scalar,
    "chi_signed": chi_signed,
    "alpha_inv_bare": alpha_inv_bare,
    "alpha_inv_dressed": alpha_inv,
    "alpha_s": alpha_s,
    "Vus": Vus,
    "Vcb": Vcb,
    "Vub": Vub,
    "J_CKM": J_ckm,
    "theta_QCD": theta_qcd,
    "eta_B": eta_baryon,
    "A_s": A_s,
    "ln10As": ln10As,
    "pmns_s12sq": pmns_s12_sq,
    "pmns_s23sq": pmns_s23_sq,
    "pmns_s13sq": pmns_s13_sq,
    "pmns_delta_deg": delta_pmns * 180.0 / pi,
    "nu_dm_ratio": nu_dm_ratio,
    "Omega_b": Omega_b,
    "Omega_c": Omega_c,
    "Omega_m": Omega_m,
    "Omega_Lambda": Omega_L,
    "Omega_dark": Omega_dark,
    "n_s": n_s,
    "running_ns": running_ns,
    "tensor_r": tensor_r,
    "tau_reio": tau_reio,
    "sigma8": sigma8,
    "x_RG": x_RG,
    "lambda_H": lambda_H,
    "yt_RG": yt_RG,
    "yt_pole": yt_pole
}

freeze_text = json.dumps(freeze, sort_keys=True, separators=(",", ":"))
freeze_hash = hashlib.sha256(freeze_text.encode("utf-8")).hexdigest()

print()
print("FREEZE HASH")
print("  Cathedral Dimensionless Core v2.7 math outputs")
print("  sha256:")
print("    " + freeze_hash)

print()
print("FINAL LINE")
print("  From chaos to closure: this is the math and output core.")

CATHEDRAL DIMENSIONLESS CORE v2.7
MATH AND OUTPUTS ONLY
Chaos -> pi-phi-e flow -> G13 -> A5 -> hidden 4+4 -> outputs

0. CORE PRIMITIVES
  D, V, N, E, F, q, G       = 3 12 13 30 20 5 60
  phi                       = 1.618033988749895
  gamma = 1/81              = 0.012345679012346
  dstar = (1-gamma)pi/Nphi  = 0.147510810159580
  dcl = D/F                 = 0.150000000000000
  Delta = dcl-dstar         = 0.002489189840420
  Cmid                      = 0.148755405079790

1. G13 GEOMETRY
  shell vertices              = 12
  shell edges                 = 30
  shell degree                = 5
  centered nodes              = 13
  center degree               = 12
  shell Laplacian spectrum:
    -0.000000 multiplicity 1
    2.763932 multiplicity 3
    6.000000 multiplicity 5
    7.236068 multiplicity 3
  centered G13 Laplacian spectrum:
    -0.000000 multiplicity 1
    3.763932 multiplicity 3
    7.000000 multiplicity 5
    8.236068 multiplicity 3
    13.000000 multiplicity 1

2. CHAOS ORIGIN 